In [ ]:
# Cell 1 — Mount Drive + runtime preflight

from google.colab import drive
drive.mount("/content/drive", force_remount=True, timeout_ms=300000)

!nvidia-smi
!free -h

import torch, psutil

assert torch.cuda.is_available(), "CUDA GPU is required."

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
ram_gb = psutil.virtual_memory().total / 1e9

print("GPU:", gpu_name)
print("GPU memory GB:", gpu_mem_gb)
print("System RAM GB:", ram_gb)

assert ("A100" in gpu_name) or gpu_mem_gb >= 35, "PE-Core-G14-448 needs A100-class GPU. Do not run on T4."
assert ram_gb >= 35, "High-RAM runtime is recommended."

print("PASS: runtime is suitable.")


In [ ]:
# Cell 2 — Prepare local test data from gallery.zip

from pathlib import Path
import shutil, zipfile, subprocess, time

SRC_DIR = Path("/content/drive/MyDrive/aic2026_data/archives/name-masked_test-set")
LOCAL_ROOT = Path("/content/aic_local")
TEST_DIR = LOCAL_ROOT / "raw" / "name-masked_test-set"
TEST_DIR.mkdir(parents=True, exist_ok=True)

src_files = {
    "gallery.zip": SRC_DIR / "gallery.zip",
    "query_index.txt": SRC_DIR / "query_index.txt",
    "query_text.json": SRC_DIR / "query_text.json",
}

for name, src in src_files.items():
    assert src.exists(), f"Missing Drive file: {src}"
    dst = TEST_DIR / name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print("SKIP existing:", dst)
    else:
        print("Copying:", src, "->", dst)
        shutil.copy2(src, dst)

def count_images(folder: Path):
    if not folder.exists():
        return 0
    cmd = (
        f'find "{folder}" -maxdepth 1 -type f '
        f'\( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" -o -iname "*.webp" \) | wc -l'
    )
    return int(subprocess.check_output(["bash", "-lc", cmd], text=True).strip())

gallery_candidates = [
    TEST_DIR / "gallery" / "gallery",
    TEST_DIR / "gallery",
    TEST_DIR,
]

current_count = max(count_images(p) for p in gallery_candidates if p.exists()) if any(p.exists() for p in gallery_candidates) else 0

if current_count != 36773:
    for old in [TEST_DIR / "gallery", TEST_DIR / "gallery_tmp"]:
        if old.exists():
            print("Removing old extracted folder:", old)
            shutil.rmtree(old)

    print("Extracting gallery.zip...")
    t0 = time.time()
    with zipfile.ZipFile(TEST_DIR / "gallery.zip", "r") as zf:
        zf.extractall(TEST_DIR)
    print(f"Extraction done in {time.time() - t0:.1f}s")
else:
    print("Gallery already extracted. Skipping extraction.")

gallery_candidates = [
    TEST_DIR / "gallery" / "gallery",
    TEST_DIR / "gallery",
    TEST_DIR,
]

for child in TEST_DIR.iterdir():
    if child.is_dir():
        gallery_candidates.append(child)

counts = sorted(
    [(count_images(p), p) for p in dict.fromkeys(gallery_candidates) if p.exists()],
    reverse=True,
)

GALLERY_DIR = counts[0][1]
n_gallery = counts[0][0]

QUERY_INDEX = TEST_DIR / "query_index.txt"
QUERY_TEXT = TEST_DIR / "query_text.json"
n_queries = len([x for x in QUERY_INDEX.read_text().splitlines() if x.strip()])

print("TEST_DIR:", TEST_DIR)
print("GALLERY_DIR:", GALLERY_DIR)
print("query count:", n_queries)
print("gallery count:", n_gallery)

assert QUERY_INDEX.exists()
assert QUERY_TEXT.exists()
assert n_queries == 1978
assert n_gallery == 36773

print("PASS: local test data is ready.")


In [ ]:
# Cell 3 — Clone repo + install PE-G14 dependencies

from pathlib import Path
import sys, os

%cd /content

!rm -rf Sim2Real-ReID
!git clone -b clean-adaption https://github.com/vquclinh/Sim2Real-ReID.git

%cd /content/Sim2Real-ReID/aic26/pe_g14

NOTEBOOK_DIR = Path(".").resolve()
UTILS_DIR = NOTEBOOK_DIR.parent / "utils"

for p in (str(UTILS_DIR), str(NOTEBOOK_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import aic_colab_utils
print("PASS: imported aic_colab_utils from:", aic_colab_utils.__file__)

PM_REPO = Path("/content/perception_models")
if not PM_REPO.exists():
    !git clone --depth 1 https://github.com/facebookresearch/perception_models.git "{PM_REPO}"

!pip install -q timm ftfy regex tokenizers einops iopath

if str(PM_REPO) not in sys.path:
    sys.path.insert(0, str(PM_REPO))

print("perception_models repo:", PM_REPO)


In [ ]:
# Cell 4 — Load queries, gallery list, and PE-G14 model

from pathlib import Path
import json, subprocess, time, shutil, zipfile
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from core.vision_encoder import pe
from core.vision_encoder import transforms as pe_transforms

TEST_DIR = Path("/content/aic_local/raw/name-masked_test-set")
QUERY_INDEX = TEST_DIR / "query_index.txt"
QUERY_TEXT = TEST_DIR / "query_text.json"

def count_images(folder: Path):
    if not folder.exists():
        return 0
    cmd = (
        f'find "{folder}" -maxdepth 1 -type f '
        f'\( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" -o -iname "*.webp" \) | wc -l'
    )
    return int(subprocess.check_output(["bash", "-lc", cmd], text=True).strip())

gallery_candidates = [
    TEST_DIR / "gallery" / "gallery",
    TEST_DIR / "gallery",
    TEST_DIR,
]

counts = sorted([(count_images(p), p) for p in gallery_candidates if p.exists()], reverse=True)
GALLERY_DIR = counts[0][1]

submission_qids = [x.strip() for x in QUERY_INDEX.read_text(encoding="utf-8").splitlines() if x.strip()]

qid_to_caption = {}
with open(QUERY_TEXT, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        qid_to_caption[obj["query_index"]] = obj["caption"]

submission_captions = [qid_to_caption[qid] for qid in submission_qids]

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
gallery_paths = sorted([p for p in GALLERY_DIR.iterdir() if p.suffix.lower() in EXTS])
gallery_ids = [p.stem for p in gallery_paths]

print("queries:", len(submission_qids))
print("gallery:", len(gallery_paths))
print("sample query:", submission_captions[0][:160])
print("sample gallery ids:", gallery_ids[:5])

assert len(submission_qids) == 1978
assert len(gallery_paths) == 36773

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print("GPU:", gpu_name)
print("GPU memory GB:", total_gb)

major, minor = torch.cuda.get_device_capability(0)
AUTOCAST_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("autocast dtype:", AUTOCAST_DTYPE)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

PE_MODEL_NAME = "PE-Core-G14-448"
print("Loading", PE_MODEL_NAME)

pe_model = pe.CLIP.from_config(PE_MODEL_NAME, pretrained=True).to(device).eval()
pe_model = pe_model.to(memory_format=torch.channels_last)

pe_preprocess = pe_transforms.get_image_transform(pe_model.image_size)
pe_tokenizer = pe_transforms.get_text_tokenizer(pe_model.context_length)

fallback_size = pe_model.image_size if isinstance(pe_model.image_size, int) else pe_model.image_size[0]
pe_fallback = pe_preprocess(Image.new("RGB", (fallback_size, fallback_size))).clone()

print("image_size:", pe_model.image_size)
print("context_length:", pe_model.context_length)


In [ ]:
# Cell 5 — Encode gallery + queries

LOCAL_OUTPUT = Path("/content/aic_local/output/zero_shot_pe_g14")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/aic2026_data/output/zero_shot_pe_g14")

LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

GALLERY_CACHE_LOCAL = LOCAL_OUTPUT / "gallery_emb.npz"
GALLERY_CACHE_DRIVE = DRIVE_OUTPUT / "gallery_emb.npz"
QUERY_CACHE_LOCAL = LOCAL_OUTPUT / "query_emb.npz"
QUERY_CACHE_DRIVE = DRIVE_OUTPUT / "query_emb.npz"

IMG_BATCH = 96 if total_gb < 60 else 256
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
TEXT_BATCH = 256

print("IMG_BATCH:", IMG_BATCH)
print("NUM_WORKERS:", NUM_WORKERS)

class GalleryDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return pe_preprocess(img), idx, True
        except Exception:
            return pe_fallback.clone(), idx, False

@torch.inference_mode()
def encode_gallery():
    if not GALLERY_CACHE_LOCAL.exists() and GALLERY_CACHE_DRIVE.exists():
        print("Restoring gallery cache from Drive...")
        shutil.copy2(GALLERY_CACHE_DRIVE, GALLERY_CACHE_LOCAL)

    if GALLERY_CACHE_LOCAL.exists():
        cached = np.load(GALLERY_CACHE_LOCAL, allow_pickle=False)
        if list(cached["ids"]) == gallery_ids and cached["emb"].shape[0] == len(gallery_ids):
            print("Reusing gallery cache:", cached["emb"].shape, cached["emb"].dtype)
            return cached["emb"]

    ds = GalleryDataset(gallery_paths)
    dl = DataLoader(
        ds,
        batch_size=IMG_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=True,
    )

    chunks = []
    ok_flags = np.zeros(len(gallery_paths), dtype=bool)
    t0 = time.time()
    seen = 0

    for tensors, idxs, oks in dl:
        tensors = tensors.to(device, non_blocking=True, memory_format=torch.channels_last)

        with torch.autocast(device_type="cuda", dtype=AUTOCAST_DTYPE):
            feats = pe_model.encode_image(tensors)
            feats = F.normalize(feats.float(), dim=-1).half().cpu().numpy()

        chunks.append(feats)
        ok_flags[idxs.numpy()] = oks.numpy().astype(bool)

        seen += tensors.size(0)
        if seen % (IMG_BATCH * 10) == 0 or seen == len(gallery_paths):
            elapsed = time.time() - t0
            rate = seen / elapsed if elapsed > 0 else 0
            eta = (len(gallery_paths) - seen) / rate if rate > 0 else 0
            print(f"encoded {seen}/{len(gallery_paths)} | {rate:.1f} img/s | elapsed {elapsed:.0f}s | ETA {eta:.0f}s")

    gallery_emb = np.concatenate(chunks, axis=0).astype(np.float16)

    print("Gallery encode done:", gallery_emb.shape, gallery_emb.dtype)
    print("failed images:", int((~ok_flags).sum()))

    np.savez_compressed(
        GALLERY_CACHE_LOCAL,
        ids=np.array(gallery_ids),
        emb=gallery_emb,
        ok=ok_flags,
    )

    shutil.copy2(GALLERY_CACHE_LOCAL, GALLERY_CACHE_DRIVE)
    print("Saved gallery cache to Drive:", GALLERY_CACHE_DRIVE)

    return gallery_emb

@torch.inference_mode()
def encode_queries():
    if not QUERY_CACHE_LOCAL.exists() and QUERY_CACHE_DRIVE.exists():
        print("Restoring query cache from Drive...")
        shutil.copy2(QUERY_CACHE_DRIVE, QUERY_CACHE_LOCAL)

    if QUERY_CACHE_LOCAL.exists():
        cached = np.load(QUERY_CACHE_LOCAL, allow_pickle=False)
        if list(cached["ids"]) == submission_qids and cached["emb"].shape[0] == len(submission_qids):
            print("Reusing query cache:", cached["emb"].shape, cached["emb"].dtype)
            return cached["emb"]

    chunks = []
    t0 = time.time()

    for start in range(0, len(submission_captions), TEXT_BATCH):
        end = min(start + TEXT_BATCH, len(submission_captions))
        texts = submission_captions[start:end]

        tokens = pe_tokenizer(texts).to(device)

        with torch.autocast(device_type="cuda", dtype=AUTOCAST_DTYPE):
            feats = pe_model.encode_text(tokens)
            feats = F.normalize(feats.float(), dim=-1).half().cpu().numpy()

        chunks.append(feats)
        print(f"encoded queries {end}/{len(submission_captions)}")

    query_emb = np.concatenate(chunks, axis=0).astype(np.float16)

    np.savez_compressed(
        QUERY_CACHE_LOCAL,
        ids=np.array(submission_qids),
        emb=query_emb,
    )

    shutil.copy2(QUERY_CACHE_LOCAL, QUERY_CACHE_DRIVE)
    print("Saved query cache to Drive:", QUERY_CACHE_DRIVE)
    print(f"Query encode done in {time.time() - t0:.1f}s")

    return query_emb

gallery_emb_np = encode_gallery()
query_emb_np = encode_queries()

print("gallery_emb_np:", gallery_emb_np.shape, gallery_emb_np.dtype)
print("query_emb_np:", query_emb_np.shape, query_emb_np.dtype)


In [ ]:
# Cell 6 — Similarity, output, validation, baseline compare

TOPK = 10
Q_CHUNK = 256

G_t = torch.from_numpy(gallery_emb_np).to(device, dtype=torch.float32)
Q_t = torch.from_numpy(query_emb_np).to(device, dtype=torch.float32)

top10_indices = np.empty((Q_t.size(0), TOPK), dtype=np.int64)

t0 = time.time()
for start in range(0, Q_t.size(0), Q_CHUNK):
    end = min(start + Q_CHUNK, Q_t.size(0))
    sims = Q_t[start:end] @ G_t.T
    _, idx = torch.topk(sims, k=TOPK, dim=1)
    top10_indices[start:end] = idx.cpu().numpy()
    print(f"processed queries {end}/{Q_t.size(0)}")

print(f"Similarity/top-k done in {time.time() - t0:.1f}s")

SUB_LOCAL = Path("/content/aic_local/submission_zero_shot")
SUB_DRIVE = Path("/content/drive/MyDrive/aic2026_data/output/submission_zero_shot")

SUB_LOCAL.mkdir(parents=True, exist_ok=True)
SUB_DRIVE.mkdir(parents=True, exist_ok=True)

ANSWER_TXT = SUB_LOCAL / "answer.txt"
ANSWER_ZIP = SUB_LOCAL / "answer.zip"

with open(ANSWER_TXT, "w", encoding="utf-8") as f:
    for row in top10_indices:
        f.write(" ".join(gallery_ids[i] for i in row) + "\n")

with zipfile.ZipFile(ANSWER_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ANSWER_TXT, arcname="answer.txt")

shutil.copy2(ANSWER_TXT, SUB_DRIVE / "answer.txt")
shutil.copy2(ANSWER_ZIP, SUB_DRIVE / "answer.zip")

lines = ANSWER_TXT.read_text(encoding="utf-8").strip().splitlines()
assert len(lines) == 1978

gallery_set = set(gallery_ids)
bad = []

for i, line in enumerate(lines):
    toks = line.split()
    if len(toks) != 10:
        bad.append((i, f"len={len(toks)}"))
    elif len(set(toks)) != 10:
        bad.append((i, "duplicate IDs"))
    elif any("." in tok for tok in toks):
        bad.append((i, "contains extension/dot"))
    else:
        missing = [tok for tok in toks if tok not in gallery_set]
        if missing:
            bad.append((i, f"unknown IDs: {missing[:3]}"))

assert not bad, f"{len(bad)} bad lines; first: {bad[:5]}"

print("PASS: answer.txt format OK")
print("answer.txt:", ANSWER_TXT)
print("answer.zip:", ANSWER_ZIP)
print("Drive output:", SUB_DRIVE)
print("first line:", lines[0])

BASELINE_ANSWER = Path("/content/Sim2Real-ReID/aic26/docs/references/baseline_answer_pe_g14.txt")

if BASELINE_ANSWER.exists():
    baseline_lines = BASELINE_ANSWER.read_text(encoding="utf-8").strip().splitlines()
    assert len(baseline_lines) == len(lines) == 1978

    exact_row_match = sum(a == b for a, b in zip(baseline_lines, lines))
    top1_match = sum(a.split()[0] == b.split()[0] for a, b in zip(baseline_lines, lines))

    print(f"Exact row match: {exact_row_match}/{len(lines)} ({100 * exact_row_match / len(lines):.2f}%)")
    print(f"Top-1 match    : {top1_match}/{len(lines)} ({100 * top1_match / len(lines):.2f}%)")

    if exact_row_match < len(lines):
        print("\nFirst different rows:")
        shown = 0
        for i, (a, b) in enumerate(zip(baseline_lines, lines)):
            if a != b:
                print("row", i)
                print("baseline:", a)
                print("new     :", b)
                shown += 1
                if shown >= 5:
                    break
else:
    print("Baseline answer not found:", BASELINE_ANSWER)


In [ ]:
# Cell 7 — Save run artifacts and metadata

from pathlib import Path
import shutil
import zipfile
import subprocess
import datetime

RUN_ID = "run_002_pe_g14_reproduce"

DRIVE_ROOT = Path("/content/drive/MyDrive/aic2026_data")
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

ANSWER_TXT = DRIVE_ROOT / "output" / "submission_zero_shot" / "answer.txt"
ANSWER_ZIP = DRIVE_ROOT / "output" / "submission_zero_shot" / "answer.zip"

GALLERY_CACHE = DRIVE_ROOT / "output" / "zero_shot_pe_g14" / "gallery_emb.npz"
QUERY_CACHE = DRIVE_ROOT / "output" / "zero_shot_pe_g14" / "query_emb.npz"

assert ANSWER_TXT.exists(), f"Missing {ANSWER_TXT}"
assert ANSWER_ZIP.exists(), f"Missing {ANSWER_ZIP}"

shutil.copy2(ANSWER_TXT, RUN_DIR / "answer.txt")
shutil.copy2(ANSWER_ZIP, RUN_DIR / "answer.zip")

with zipfile.ZipFile(RUN_DIR / "answer.zip", "r") as zf:
    names = zf.namelist()
    assert names == ["answer.txt"], f"answer.zip should contain only answer.txt, got: {names}"

lines = (RUN_DIR / "answer.txt").read_text(encoding="utf-8").strip().splitlines()
assert len(lines) == 1978, f"Expected 1978 lines, got {len(lines)}"

bad = []
for i, line in enumerate(lines):
    toks = line.split()
    if len(toks) != 10:
        bad.append((i, f"len={len(toks)}"))
    elif len(set(toks)) != 10:
        bad.append((i, "duplicate IDs"))
    elif any("." in tok for tok in toks):
        bad.append((i, "contains dot/extension"))

assert not bad, f"Bad answer format: {bad[:5]}"

BASELINE = Path("/content/Sim2Real-ReID/aic26/docs/references/baseline_answer_pe_g14.txt")

comparison_text = "Baseline answer not found; comparison skipped.\n"
exact_row_match = None
top1_match = None

if BASELINE.exists():
    baseline_lines = BASELINE.read_text(encoding="utf-8").strip().splitlines()
    assert len(baseline_lines) == len(lines) == 1978

    exact_row_match = sum(a == b for a, b in zip(baseline_lines, lines))
    top1_match = sum(a.split()[0] == b.split()[0] for a, b in zip(baseline_lines, lines))

    comparison_text = (
        f"Exact row match: {exact_row_match}/1978 ({100 * exact_row_match / 1978:.2f}%)\n"
        f"Top-1 match    : {top1_match}/1978 ({100 * top1_match / 1978:.2f}%)\n"
    )

    if exact_row_match < 1978:
        comparison_text += "\nFirst different rows:\n"
        shown = 0
        for i, (a, b) in enumerate(zip(baseline_lines, lines)):
            if a != b:
                comparison_text += f"\nrow {i}\n"
                comparison_text += f"baseline: {a}\n"
                comparison_text += f"new     : {b}\n"
                shown += 1
                if shown >= 5:
                    break

(RUN_DIR / "comparison_with_baseline.txt").write_text(comparison_text, encoding="utf-8")

def safe_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception as e:
        return f"ERROR: {e}"

repo_commit = safe_cmd("git -C /content/Sim2Real-ReID rev-parse HEAD")
repo_branch = safe_cmd("git -C /content/Sim2Real-ReID rev-parse --abbrev-ref HEAD")
gpu_info = safe_cmd("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
python_version = safe_cmd("python --version")

run_info = (
    f"# {RUN_ID}\n"
    "\n"
    f"Date: {datetime.datetime.now().isoformat(timespec='seconds')}\n"
    "\n"
    "## Purpose\n"
    "\n"
    "Reproduce the teammate PE-G14 zero-shot baseline after moving the pipeline into the cleaned forked repository.\n"
    "\n"
    "## Repository\n"
    "\n"
    f"Branch: {repo_branch}  \n"
    f"Commit: {repo_commit}\n"
    "\n"
    "## Method\n"
    "\n"
    "Model: PE-Core-G14-448  \n"
    "Training: none  \n"
    "Fine-tuning: none  \n"
    "Ensemble: none  \n"
    "Re-ranking: none  \n"
    "\n"
    "## Dataset\n"
    "\n"
    "Dataset: AIC/ECCV 2026 Track 4 name-masked test set  \n"
    "Query count: 1,978  \n"
    "Gallery count: 36,773  \n"
    "\n"
    "## Runtime\n"
    "\n"
    "GPU:\n"
    f"`{gpu_info}`\n"
    "\n"
    "Python:\n"
    f"`{python_version}`\n"
    "\n"
    "## Outputs\n"
    "\n"
    "Run folder:\n"
    f"`{RUN_DIR}`\n"
    "\n"
    "Files:\n"
    "- `answer.txt`\n"
    "- `answer.zip`\n"
    "- `comparison_with_baseline.txt`\n"
    "- `run_info.md`\n"
    "\n"
    "Embedding cache files are stored separately:\n"
    f"- `{GALLERY_CACHE}`\n"
    f"- `{QUERY_CACHE}`\n"
    "\n"
    "## Format Validation\n"
    "\n"
    f"answer.txt rows: {len(lines)}  \n"
    "Each row has 10 IDs: yes  \n"
    "No image extensions: yes  \n"
    "answer.zip contains only answer.txt: yes  \n"
    "\n"
    "## Baseline Comparison\n"
    "\n"
    f"{comparison_text}\n"
)

(RUN_DIR / "run_info.md").write_text(run_info, encoding="utf-8")

print("PASS: run artifacts saved.")
print("RUN_DIR:", RUN_DIR)
print("answer.txt:", RUN_DIR / "answer.txt")
print("answer.zip:", RUN_DIR / "answer.zip")
print("comparison:", RUN_DIR / "comparison_with_baseline.txt")
print("run_info.md:", RUN_DIR / "run_info.md")
print()
print(comparison_text)
print("Cache files:")
print("gallery_emb.npz:", GALLERY_CACHE.exists(), GALLERY_CACHE)
print("query_emb.npz  :", QUERY_CACHE.exists(), QUERY_CACHE)
